# Experiment 2A: DataBoost with Verbalized Sampling

**Aim:** Train a baseline ModernBERT model, then retrain with VS-augmented data targeting misclassified validation samples. Compare baseline vs DataBoosted accuracy.

**Key difference from NB02:** Instead of LLM paraphrasing at runtime, we use a pre-generated Verbalized Sampling (VS) augmentation dataset that targets the specific confusion patterns found in NB02's error analysis (POSITIVE→NEUTRAL, NEUTRAL→POSITIVE, NEGATIVE→NEUTRAL).

## 1. Setup & Installation

In [ ]:
# setup

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn peft accelerate transformers

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import torch
import torch.nn.functional as F
from transformers import (
    TrainingArguments, Trainer, AutoModelForSequenceClassification,
    AutoTokenizer, training_args,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm import tqdm
import os

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5

## 2. Data Preparation

In [ ]:
ds = load_dataset("neoyipeng/financial_reasoning_aggregated")

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

## 3. Baseline Training

In [ ]:
def train_model(train_dataset, val_dataset, output_dir="trainer_output", epochs=10):
    model = AutoModelForSequenceClassification.from_pretrained(
        "answerdotai/ModernBERT-base",
        num_labels=NUM_CLASSES,
        torch_dtype=torch.float32,
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()
    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    model = model.cuda()
    model.print_trainable_parameters()

    def tokenize_function(examples):
        return tokenizer(examples["text"])

    train_tok = train_dataset.map(tokenize_function, batched=True)
    val_tok = val_dataset.map(tokenize_function, batched=True)

    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        args=TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            fp16=True,
            bf16=False,
            optim=training_args.OptimizerNames.ADAMW_TORCH,
            learning_rate=2e-4,
            weight_decay=0.001,
            lr_scheduler_type="cosine",
            seed=3407,
            num_train_epochs=epochs,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_strategy="epoch",
            eval_strategy="epoch",
            logging_strategy="epoch",
            gradient_checkpointing=True,
            report_to="none",
        ),
        compute_metrics=lambda eval_pred: {
            "accuracy": accuracy_score(
                eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
            )
        },
    )

    trainer.train()
    model = model.cuda().eval()
    return model, tokenizer

In [ ]:
print("Training BASELINE model...")
baseline_model, tokenizer = train_model(
    ds["train"], ds["validation"], output_dir="trainer_output_baseline"
)

## 4. Baseline Evaluation

In [ ]:
def run_inference(model, tokenizer, texts, batch_size=32):
    all_preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Inference"):
            batch = texts[i : i + batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
            )
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

In [ ]:
# Baseline on validation
val_texts = ds["validation"]["text"]
val_labels = np.argmax(ds["validation"]["labels"], axis=1)
val_preds = run_inference(baseline_model, tokenizer, val_texts)
val_acc = accuracy_score(val_labels, val_preds)
val_f1 = f1_score(val_labels, val_preds, average="macro")
print(f"Baseline validation — Accuracy: {val_acc:.4f}  Macro F1: {val_f1:.4f}")
print(classification_report(val_labels, val_preds, target_names=LABEL_NAMES))

# Save validation errors
errors = []
for i in range(len(val_texts)):
    if val_preds[i] != val_labels[i]:
        errors.append({
            "text": val_texts[i],
            "true_label": int(val_labels[i]),
            "true_label_name": LABEL_NAMES[val_labels[i]],
            "pred_label": int(val_preds[i]),
            "pred_label_name": LABEL_NAMES[val_preds[i]],
        })
error_df = pd.DataFrame(errors)
error_df.to_csv("validation_errors.csv", index=False)
print(f"\nMisclassified: {len(errors)} / {len(val_texts)} ({len(errors)/len(val_texts):.1%})")

In [ ]:
# Baseline on FPB
fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]

baseline_fpb50_preds = run_inference(baseline_model, tokenizer, fpb_50["sentence"])
baseline_fpb50_acc = accuracy_score(fpb_50["label"], baseline_fpb50_preds)
baseline_fpb50_f1 = f1_score(fpb_50["label"], baseline_fpb50_preds, average="macro")
print(f"Baseline FPB 50agree — Accuracy: {baseline_fpb50_acc:.4f}  Macro F1: {baseline_fpb50_f1:.4f}")
print(classification_report(fpb_50["label"], baseline_fpb50_preds, target_names=LABEL_NAMES))

baseline_fpball_preds = run_inference(baseline_model, tokenizer, fpb_all["sentence"])
baseline_fpball_acc = accuracy_score(fpb_all["label"], baseline_fpball_preds)
baseline_fpball_f1 = f1_score(fpb_all["label"], baseline_fpball_preds, average="macro")
print(f"Baseline FPB allAgree — Accuracy: {baseline_fpball_acc:.4f}  Macro F1: {baseline_fpball_f1:.4f}")
print(classification_report(fpb_all["label"], baseline_fpball_preds, target_names=LABEL_NAMES))

In [ ]:
# Baseline on aggregated test set
test_texts_agg = ds["test"]["text"]
test_labels_agg = np.argmax(ds["test"]["labels"], axis=1)
baseline_test_preds = run_inference(baseline_model, tokenizer, test_texts_agg)
baseline_test_acc = accuracy_score(test_labels_agg, baseline_test_preds)
baseline_test_f1 = f1_score(test_labels_agg, baseline_test_preds, average="macro")
print(f"Baseline Test Set — Accuracy: {baseline_test_acc:.4f}  Macro F1: {baseline_test_f1:.4f}")
print(classification_report(test_labels_agg, baseline_test_preds, target_names=LABEL_NAMES))

## 5. Load Verbalized Sampling Augmentation Data

Pre-generated VS-CoT augmented data targeting 82 misclassified validation samples.
410 augmented texts (5 per seed) spanning earnings calls, analyst notes, news headlines,
press releases, social media, and SEC filings.

In [ ]:
import base64, gzip, io

VS_DATA_B64 = "..."

# Decode and load VS augmentation data
vs_csv = gzip.decompress(base64.b64decode(VS_DATA_B64)).decode()
vs_df = pd.read_csv(io.StringIO(vs_csv))
print(f"Loaded {len(vs_df)} VS-augmented samples")
print(f"\nBy label:")
print(vs_df["label_name"].value_counts().to_string())
print(f"\nBy confusion type targeted:")
print(vs_df["confusion_type"].value_counts().to_string())

## 6. Create Augmented Training Set

In [ ]:
# Convert VS augmented data to HF dataset with one-hot labels
aug_texts = vs_df["text"].tolist()
aug_labels_int = vs_df["label"].tolist()
aug_labels = [np.eye(NUM_CLASSES)[lbl].tolist() for lbl in aug_labels_int]

aug_ds = Dataset.from_dict({"text": aug_texts, "labels": aug_labels})

# Combine original training data + VS augmentations
augmented_train = concatenate_datasets([ds["train"], aug_ds]).shuffle(seed=42)

print(f"Original train size:  {len(ds['train']):,}")
print(f"VS augmentation size: {len(aug_ds):,}")
print(f"Augmented train size: {len(augmented_train):,}")
print(f"Augmentation ratio:   {len(aug_ds)/len(ds['train']):.1%}")

In [ ]:
# Save datasets for downstream use
aug_ds.save_to_disk("databoost_paraphrases")
vs_df.to_csv("databoost_paraphrases.csv", index=False)
augmented_train.save_to_disk("augmented_train_dataset")
print(f"Saved VS paraphrases and augmented dataset to disk")

## 7. Retrain on Augmented Data

In [ ]:
# Free baseline model memory
del baseline_model
torch.cuda.empty_cache()

print("Training DATABOOSTED model (with VS augmentation)...")
boosted_model, tokenizer = train_model(
    augmented_train, ds["validation"], output_dir="trainer_output_boosted"
)

## 8. DataBoosted Evaluation

In [ ]:
# DataBoosted on validation
boosted_val_preds = run_inference(boosted_model, tokenizer, val_texts)
boosted_val_acc = accuracy_score(val_labels, boosted_val_preds)
boosted_val_f1 = f1_score(val_labels, boosted_val_preds, average="macro")
print(f"DataBoosted validation — Accuracy: {boosted_val_acc:.4f}  Macro F1: {boosted_val_f1:.4f}")
print(classification_report(val_labels, boosted_val_preds, target_names=LABEL_NAMES))

In [ ]:
# DataBoosted on FPB
boosted_fpb50_preds = run_inference(boosted_model, tokenizer, fpb_50["sentence"])
boosted_fpb50_acc = accuracy_score(fpb_50["label"], boosted_fpb50_preds)
boosted_fpb50_f1 = f1_score(fpb_50["label"], boosted_fpb50_preds, average="macro")
print(f"DataBoosted FPB 50agree — Accuracy: {boosted_fpb50_acc:.4f}  Macro F1: {boosted_fpb50_f1:.4f}")
print(classification_report(fpb_50["label"], boosted_fpb50_preds, target_names=LABEL_NAMES))

boosted_fpball_preds = run_inference(boosted_model, tokenizer, fpb_all["sentence"])
boosted_fpball_acc = accuracy_score(fpb_all["label"], boosted_fpball_preds)
boosted_fpball_f1 = f1_score(fpb_all["label"], boosted_fpball_preds, average="macro")
print(f"DataBoosted FPB allAgree — Accuracy: {boosted_fpball_acc:.4f}  Macro F1: {boosted_fpball_f1:.4f}")
print(classification_report(fpb_all["label"], boosted_fpball_preds, target_names=LABEL_NAMES))

In [ ]:
# DataBoosted on aggregated test set
boosted_test_preds = run_inference(boosted_model, tokenizer, test_texts_agg)
boosted_test_acc = accuracy_score(test_labels_agg, boosted_test_preds)
boosted_test_f1 = f1_score(test_labels_agg, boosted_test_preds, average="macro")
print(f"DataBoosted Test Set — Accuracy: {boosted_test_acc:.4f}  Macro F1: {boosted_test_f1:.4f}")
print(classification_report(test_labels_agg, boosted_test_preds, target_names=LABEL_NAMES))

## 9. Comparison: Baseline vs DataBoosted (VS)

In [ ]:
comparison = pd.DataFrame([
    # Published baselines (in-domain FPB train/test splits)
    {"Model": "LSTM+ELMo *",        "Split": "FPB 50agree",  "Accuracy": 0.7500, "Macro F1": 0.7000},
    {"Model": "ULMFit *",           "Split": "FPB 50agree",  "Accuracy": 0.8300, "Macro F1": 0.7900},
    {"Model": "ProsusAI/finbert *", "Split": "FPB 50agree",  "Accuracy": 0.8600, "Macro F1": 0.8400},
    {"Model": "FinBERT-FinVocab *", "Split": "FPB 50agree",  "Accuracy": 0.8720, "Macro F1": None},
    {"Model": "ProsusAI/finbert *", "Split": "FPB allAgree", "Accuracy": 0.9700, "Macro F1": 0.9500},
    # Our models (FPB held out from training)
    {"Model": "Baseline (ours)",         "Split": "Validation",   "Accuracy": val_acc,             "Macro F1": val_f1},
    {"Model": "DataBoosted-VS (ours)",   "Split": "Validation",   "Accuracy": boosted_val_acc,     "Macro F1": boosted_val_f1},
    {"Model": "Baseline (ours)",         "Split": "Test Set",     "Accuracy": baseline_test_acc,   "Macro F1": baseline_test_f1},
    {"Model": "DataBoosted-VS (ours)",   "Split": "Test Set",     "Accuracy": boosted_test_acc,    "Macro F1": boosted_test_f1},
    {"Model": "Baseline (ours)",         "Split": "FPB 50agree",  "Accuracy": baseline_fpb50_acc,  "Macro F1": baseline_fpb50_f1},
    {"Model": "DataBoosted-VS (ours)",   "Split": "FPB 50agree",  "Accuracy": boosted_fpb50_acc,   "Macro F1": boosted_fpb50_f1},
    {"Model": "Baseline (ours)",         "Split": "FPB allAgree", "Accuracy": baseline_fpball_acc, "Macro F1": baseline_fpball_f1},
    {"Model": "DataBoosted-VS (ours)",   "Split": "FPB allAgree", "Accuracy": boosted_fpball_acc,  "Macro F1": boosted_fpball_f1},
])

print("=" * 85)
print("DATABOOST (Verbalized Sampling) RESULTS")
print("=" * 85)

for split in ["Validation", "Test Set", "FPB 50agree", "FPB allAgree"]:
    subset = comparison[comparison["Split"] == split].copy()
    subset = subset.sort_values("Accuracy", ascending=False)
    print(f"\n--- {split} ---")
    print(subset[["Model", "Accuracy", "Macro F1"]].to_string(index=False, float_format="%.4f"))

print(f"\n{'=' * 85}")
print("DATABOOST-VS IMPROVEMENT (Baseline -> DataBoosted)")
print(f"{'=' * 85}")
print(f"  Validation:   {boosted_val_acc - val_acc:+.4f} accuracy  |  {boosted_val_f1 - val_f1:+.4f} macro F1")
print(f"  Test Set:     {boosted_test_acc - baseline_test_acc:+.4f} accuracy  |  {boosted_test_f1 - baseline_test_f1:+.4f} macro F1")
print(f"  FPB 50agree:  {boosted_fpb50_acc - baseline_fpb50_acc:+.4f} accuracy  |  {boosted_fpb50_f1 - baseline_fpb50_f1:+.4f} macro F1")
print(f"  FPB allAgree: {boosted_fpball_acc - baseline_fpball_acc:+.4f} accuracy  |  {boosted_fpball_f1 - baseline_fpball_f1:+.4f} macro F1")
print(f"\nVS augmentation added {len(aug_ds)} samples ({len(aug_ds)/len(ds['train']):.1%} of original train)")
print(f"Targeting {len(vs_df['confusion_type'].unique())} confusion patterns from error analysis")
print("\n* Published baselines trained/tested on in-domain FPB splits (Araci 2019, Yang et al. 2020)")
print("  Our models never saw FPB during training — stricter held-out evaluation.")